In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math
from pydantic import BaseModel, Field, model_validator 


In [ ]:
class GPT3Config(BaseModel):
    vocab_size: int = Field(default=50257, gt=0, description="Vocabulary size") # 256 is the base token + 50000 BPE (Byte Pair Encoding) + 1 special token (end-of-text token) = 50257
    context_length: int = Field(default=1024, gt=0, description="Max context length/Block size")
    hidden_size: int = Field(default=12288, gt=0, description="model/hidden dimension aka d_model")  # embed_dim == hidden_size == n_embed == d_model
    n_layer: int = Field(default=12, gt=0, description="number of stacked decoder blocks")
    num_head: int = Field(default=96, gt=0, description="number of attention heads")
    intermediate_step: int = Field(default=3072, gt=0, description="FFN inner dimension")
    dropout: float = Field(default=0.1, ge=0.0, description="Residual dropout")
    attention_dropout: float = Field(default=0.1, ge=0.0, description="attention dropout")
    embedding_dropout: float = Field(default=0.1, ge=0.0, description="embedding dropout")

model_config = {"frozen": True}

@model_validator(mode = "after")
def check_head_division(self):
    if self.hidden_size % self.num_head != 0:
        raise ValueError(
            f"hidden_size ({self.hidden_size}) must be devisible by num_head ({self.num_head})"
            f"got remainder {self.hidden_size % self.num_head}"
        )
    return self 


In [ ]:
class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.num_head = config.num_head
        self.hidden_size = config.hidden_size 
        self.head_dim = self.hidden_size // self.num_head 

        self.qkv_proj = nn.Linear(config.hidden_size, 3 * config.hidden_size)
        self.out_proj = nn.Linear(config.hidden_size, config.hidden_size)
        self.out_proj.RESIDUAL_SCALE_INIT = True  # control signal variance and prevent vanishing or exploding gradient

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(torch.ones(config.context_length, config.context_length))

        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv_proj(x)
        q, k , v = qkv.split(C, dim = 2)

        q = q.view(B, T, self.num_head, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_head, self.head_dim).transpose(1,2)

        attention = q @ k.tranpose(-2, -1) / math.sqrt(self.head_dim) # [B, H, T, D] @ [B. H, D, T] --> [B, H, T, T]
        attention = attention.masked_fill(self.causal_mask[:, :, T, T] == 0, float("-inf"))
        attention = self.attention_dropout(F.softmax(attention, dim = -1))

        score = attention @ v # [B, H, T, T] @ [B, H, T, D ] --> [B, H, T, D]
        score = score.transpose(1, 2).contiguous().view(B, T, C)

        return self.residual_dropout(self.out_proj(score))


In [ ]:
class PositionWiseFNN(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_step) 
        self.fc2 = nn.Linear(config.intermediate_step, config.hidden_size)

        self.gelu = nn.GELU(approximate= "tanh" )
        self.dropout = nn.Dropoout(config.dropout)

        self.fc2.RESIDUAL_SCALE_INIT = True 

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        return self.dropout(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.hidden_size)
        self.attention = MaskedMultiHeadSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.hidden_size)
        self.ffn = PositionWiseFNN(config)


    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


In [ ]:
class GPT3(nn.Module):
    def __init__(self, config: GPT3Config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_embedding = nn.Embedding(config.context_length, config.hidden_size)

        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])

        self.ln3 = nn.LayerNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias = False)

        self.lm_head.weigths = self.token_embedding.weight
        self.apply(self._init_weigths) #weight tying

        for module in self.modules():
            if getattr(module, "RESIDUAL_SCALE_INIT", False):
                nn.init.normal_(
                    module.weight, mean = 0.0, std = 0.02 / math.sqrt(2 * config.n_layer)
                ) # More layers -> smaller nudge per layer then total growth stays under control

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None: 
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

    def forward(self, idx, targets = None):
        # idx     : (B, T) LongTensor of token ids
        # targets : (B, T) LongTensor of next-token ids
        B, T = idx.shape
        assert T <= self.config.context_length, "sequence longer than context_length"

        pos_ids = torch.arrange(0, T, dtype = torch.long, device = idx.device)

        tok = self.token_embedding(idx)
        pos = self.position_embedding(idx)
        x = self.embedding_dropout(pos + tok)

        for block in self.blocks:
            x = block(x) # going through the transformer block

        x = self.ln3(x)
        logits = self.lm_head(x)

        loss = None

        if targets is not None: 
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )

        return logits, loss 

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature = 1.0, top_k = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")


            probs = F.softmax(logits, dim = -1)
            next_id = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, next_id), dim = -1)

        return idx


    def num_params(self, non_embedding = False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.position_embed.weight.numel()

        return n 

if __name__ == "__main__":
    torch.manual_seeD(42)
    config = GPT3Config
    model = GPT3(config)


    print(f"Total parameters: {model.num_params():,}")

    B, T = 2, 32

    dummy_idx = torch.randint(0, config.vocab_size, (B, T))
    dummy_target = torch.randint(0, config.vocab_size, (B, T))

    logits, loss = model(dummy_idx, dummy_target)
    print("Logits shape:", tuple(logits.shape))
    print("Loss:", loss.item())

    generated = model.generate(dummy_idx, max_new_tokens = 5, top_k = 10)
    print("Generated shape:", tuple(generated.shape))

    print("Training Demo:")

    text = ( "the quick brown fox jumps over the lazy dog. "
            "she sells seashells by the seashore. "
            "to be or not to be, that is the question. ") * 50

    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    def encode(s):
        return [stoi[c] for c in s]
    def decode(ids):
        return "".join(itos[i] for i in ids)

    data = torch.tensor(encode(text), dtype = torch.long)

    small_config = GPT3Config(
        vocab_size = vocab_size, 
        context_length = 64,
        hidden_size = 128,
        n_layer = 4, 
        num_head = 4, 
        intermediate_step = 512,
        dropout = 0.1,
    )

    small_model = GPT3(small_config)
    print(f"Model parameters: {small_model.num_params():,}")


    block_size = small_config.context_length
    batch_size = 32

    def get_batch():
        ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
        x = torch.stack([data[i:i + block_size] for i in ix])
        y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
        return x, y

    optimizer = torch.optim.AdamW(small_model.parameters(), lr = 3e-4)

    small_model.train()
    for step in range(300):
        xb, yb = get_batch()
        logits, loss = small_model(xb, yb)

        optimizer.zero_grad(set_to_none = True)
        loss.backward()
        optimizer.step()
        if step % 50 == 0 or step == 299:
            print(f"Step {step:4d} | Loss {loss.item():.4f}")

    small_model.eval()
    context = torch.tensor([encode("the ")], dtype = torch.long)
    out = small_model.generate(context, max_new_tokens = 80, temperature = 0.8, top_k = 10)
    print("\nSample Generation:")
    print(decode(out[0].tolist()))

